# LangChain Content Studio

A practical LangChain workflow for turning a source article into publication-ready content with Google Gemini. The notebook demonstrates prompt composition, LCEL chains, structured output, and text-based image prompt generation.

## Workflow

1. Configure the Gemini API and create deterministic and creative model clients.
2. Generate a clear article title.
3. Write an SEO-friendly description from the article and title.
4. Improve one paragraph and return constructive editorial feedback.
5. Create a concise prompt for an external image-generation tool.

> **Prerequisite:** Set `GOOGLE_API_KEY` in the environment or enter it when prompted. Keep API keys out of notebooks and source control.

In [4]:
!pip install -qU \
langchain-core \
langchain-google-genai \
langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1. Install Dependencies

Install the LangChain integrations used in this workflow. In a reusable project, pin compatible versions in a dependency file instead of installing from inside the notebook.

In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or getpass(
    "Enter Gemini API Key: "
)

gemini_model = "gemini-2.5-flash"

Enter Gemini API Key: ··········


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    temperature = 0.0,
    model=gemini_model,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

creative_llm = ChatGoogleGenerativeAI(
    temperature = 0.7,
    model=gemini_model,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)


## 2. Configure Gemini

Two chat model clients are created for different responsibilities:

- `llm` uses a deterministic temperature for consistent descriptions and image prompts.
- `creative_llm` uses a higher temperature for title ideas and editorial rewrites.

The notebook reads the API key from `GOOGLE_API_KEY` and only falls back to an interactive prompt when it is not already available.

In [8]:
article = """
### The Rise of Retrieval-Augmented Generation in Modern AI.
Retrieval-Augmented Generation, commonly known as RAG, is becoming one of the most important techniques for building reliable AI applications.
Unlike traditional language models that depend primarily on knowledge learned during training, RAG systems can retrieve relevant information from external sources and use it as context when generating a response.
This makes it possible to build AI assistants that work with private documents, company knowledge bases, research papers, and frequently updated information.
Modern RAG systems are also moving beyond simple vector search by combining keyword search with semantic retrieval, reranking relevant documents, transforming user queries, and evaluating the quality of retrieved information and generated answers.
Observability is another important part of production RAG systems because developers need to understand what information was retrieved, how the model used it, how long each step took, and where errors occurred.
As these techniques continue to improve, RAG is becoming increasingly useful for applications such as enterprise search, customer support, research assistants, document analysis, and knowledge management.
"""

## 3. Define the Source Article

This sample article gives every downstream chain a shared input. Replace `article` with a document loaded from your own application, file store, or retrieval pipeline.

In [10]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

system_prompt = SystemMessagePromptTemplate.from_template(
    "You are an Ai Assistant that helps generate article titles"
)

user_prompt = HumanMessagePromptTemplate.from_template(
    """You are tasked with creating a name for a article.

    -----


    The article is here for you to examine

    ------

    {article}

    ------

    The name should be based on the content in the article.
    Be ceative, but make sure the name are clear, catchy.
    and relevant to the theme of the article.

    ------

    Only output the article name, no other explanation or text can be provided.""",
    input_variable = ['article']
)

## 4. Generate a Title and Description

The first LCEL chain turns the article into a title. A second chain uses both the original article and generated title to produce an SEO-friendly description. The second chain demonstrates how one model output can become the next chain's input.

In [12]:
user_prompt.format(article = "TEST STRING")

HumanMessage(content='You are tasked with creating a name for a article.\n    The article is here for you to examine TEST STRING\n\n    The name should be based on the content in the article.\n    Be ceative, but make sure the name are clear, catchy.\n    and relevant to the theme of the article.\n\n    Only output the article name, no other explanation or text can be provided.', additional_kwargs={}, response_metadata={})

In [14]:
from langchain_core.prompts import ChatPromptTemplate

first_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

In [15]:
print(first_prompt.format(article = "TEST STRING"))

System: You are an Ai Assistant that helps generate article titles
Human: You are tasked with creating a name for a article.
    The article is here for you to examine TEST STRING

    The name should be based on the content in the article.
    Be ceative, but make sure the name are clear, catchy.
    and relevant to the theme of the article.

    Only output the article name, no other explanation or text can be provided.


In [17]:
chain_one = (
    {"article" : lambda x : x["article"]}
    | first_prompt
    | creative_llm
    | {"article_title" : lambda x : x.content}
)

In [20]:
article_title_msg = chain_one.invoke({"article" : article})
article_title_msg

{'article_title': 'RAG: The Contextual AI Revolution'}

In [21]:
second_user_prompt = HumanMessagePromptTemplate.from_template(
    """You are tasked with creating a description for
the article. The article is here for you to examine:

---

{article}

---

Here is the article title '{article_title}'.

Output the SEO friendly article description. Do not output
anything other than the description.""",
    input_variables=["article", "article_title"]
)

second_prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    second_user_prompt
])

In [22]:
chain_two = (
    {
        "article": lambda x: x["article"],
        "article_title": lambda x: x["article_title"]
    }
    | second_prompt
    | llm
    | {"summary": lambda x: x.content}
)

In [23]:
article_description_msg = chain_two.invoke({
    "article": article,
    "article_title": article_title_msg["article_title"]
})
article_description_msg

{'summary': 'Uncover the power of Retrieval-Augmented Generation (RAG), the contextual AI revolution making AI applications more reliable. Learn how RAG systems retrieve external information from private documents and knowledge bases, enhancing language models for enterprise search, customer support, and more. Explore advanced RAG techniques and observability.'}

In [24]:
third_user_prompt = HumanMessagePromptTemplate.from_template(
    """You are tasked with creating a new paragraph for the
article. The article is here for you to examine:

---

{article}

---

Choose one paragraph to review and edit. During your edit
ensure you provide constructive feedback to the user so they
can learn where to improve their own writing.""",
    input_variables=["article"]
)

# prompt template 3: creating a new paragraph for the article
third_prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    third_user_prompt
])

## 5. Review and Improve a Paragraph

The editorial chain uses Pydantic structured output so the result is returned as three predictable fields: the selected original paragraph, an improved version, and actionable feedback. This makes the output easier to render in an application or validate in tests.

In [25]:
from pydantic import BaseModel, Field

class Paragraph(BaseModel):
    original_paragraph: str = Field(description="The original paragraph")
    edited_paragraph: str = Field(description="The improved edited paragraph")
    feedback: str = Field(description=(
        "Constructive feedback on the original paragraph"
    ))

structured_llm = creative_llm.with_structured_output(Paragraph)

In [26]:
chain_three = (
    {"article": lambda x: x["article"]}
    | third_prompt
    | structured_llm
    | {
        "original_paragraph": lambda x: x.original_paragraph,
        "edited_paragraph": lambda x: x.edited_paragraph,
        "feedback": lambda x: x.feedback
    }
)

In [27]:
out = chain_three.invoke({"article": article})
out

{'original_paragraph': 'Retrieval-Augmented Generation, commonly known as RAG, is becoming one of the most important techniques for building reliable AI applications. Unlike traditional language models that depend primarily on knowledge learned during training, RAG systems can retrieve relevant information from external sources and use it as context when generating a response. This makes it possible to build AI assistants that work with private documents, company knowledge bases, research papers, and frequently updated information. Modern RAG systems are also moving beyond simple vector search by combining keyword search with semantic retrieval, reranking relevant documents, transforming user queries, and evaluating the quality of retrieved information and generated answers. Observability is another important part of production RAG systems because developers need to understand what information was retrieved, how the model used it, how long each step took, and where errors occurred. As 

In [35]:
from langchain_core.prompts import PromptTemplate

image_prompt = PromptTemplate(
    input_variables=["article"],
    template=(
        "Generate a concise image-generation prompt under 500 characters "
        "based on the following article. Do not include explanations. "
        "Return only the image prompt.\n\n"
        "Article: {article}"
    )
)

# This chain uses the 'llm' defined earlier (ChatGoogleGenerativeAI) to generate a text prompt
image_generation_chain = image_prompt | llm | (lambda x: x.content)

## 6. Create an Image-Generation Prompt

The final chain converts the article into a concise text prompt for an image model. This notebook does not create an image; pass `generated_image_description` to a compatible image-generation service if visual output is needed.

In [36]:
chain_four = image_generation_chain

In [37]:
generated_image_description = chain_four.invoke({"article": article})
print(f"Gemini-generated image description: {generated_image_description}")
print("\nNote: The Gemini API can generate a textual description for an image, but it does not directly create the image itself. You would need to use an external image generation service (like DALL-E, Stable Diffusion, etc.) with this description if you want to create an actual image.")

Gemini-generated image description: Visualizing RAG: A central AI brain connected by glowing data conduits to a swirling galaxy of external knowledge (digital documents, databases, web). Information flows in, is processed, and then precise, context-aware answers emanate from the AI. Emphasize the dynamic retrieval and augmented generation process for intelligent AI applications.

Note: The Gemini API can generate a textual description for an image, but it does not directly create the image itself. You would need to use an external image generation service (like DALL-E, Stable Diffusion, etc.) with this description if you want to create an actual image.
